## 00 — Bounding Boxes

We have four simplified LOD files. Each still contains thousands of features spread across the globe. When a user is looking at Western Europe at zoom 8, there is no reason to send Siberian railroads to the renderer.

The first tool for eliminating invisible features is the **bounding box** — the smallest axis-aligned rectangle that fully contains a geometry.

This notebook covers:
1. What a bounding box is and how it is stored
2. How to compute one from a feature's coordinates
3. Why the railroad dataset already has them — and what to do with that

## What Is a Bounding Box?

An **axis-aligned bounding box (AABB)** is defined by four values:

```
[lon_min, lat_min, lon_max, lat_max]
```

This is also the GeoJSON `bbox` convention. Every GeoJSON object can optionally carry a `bbox` field with this exact format.

```
lat_max  ┌───────────────┐
         │               │
         │   feature     │
         │               │
lat_min  └───────────────┘
      lon_min          lon_max
```

The bounding box does not describe the shape of the feature — only its **extent**. Two very different shapes can have identical bounding boxes.

## The Railroad Dataset Already Has Bounding Boxes

Recall from Module 00 that each feature in `ne_10m_railroads.geojson` has a `bbox` key.

Let's inspect it.

In [1]:
import json
from pathlib import Path

data_path = Path("../../data/ne_10m_railroads.geojson")
with open(data_path) as f:
    railroads = json.load(f)

feature = railroads["features"][0]

print("Feature keys:", list(feature.keys()))
print("bbox:", feature["bbox"])
print()
print("Format: [lon_min, lat_min, lon_max, lat_max]")

Feature keys: ['type', 'properties', 'bbox', 'geometry']
bbox: [30.730275, 69.448054, 30.782502, 69.461111]

Format: [lon_min, lat_min, lon_max, lat_max]


The `bbox` field is precomputed and trustworthy for the raw data.

However, our LOD files were written by the pipeline in the previous module — without `bbox` fields. So we need to be able to **compute** a bounding box from coordinates ourselves.

## Computing a Bounding Box

Given a list of `[lon, lat]` coordinate pairs, the bounding box is simply the min and max of each axis.

In [2]:
def feature_bbox(feature):
    """
    Compute [lon_min, lat_min, lon_max, lat_max] for a GeoJSON LineString feature.
    """
    coords = feature["geometry"]["coordinates"]
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    return [min(lons), min(lats), max(lons), max(lats)]

In [3]:
# Verify our result matches the precomputed bbox
computed  = feature_bbox(feature)
precomputed = feature["bbox"]

print("Computed:    ", computed)
print("Precomputed: ", precomputed)
print("Match:", computed == precomputed)

Computed:     [30.730275, 69.448054, 30.782502, 69.461111]
Precomputed:  [30.730275, 69.448054, 30.782502, 69.461111]
Match: True


## Visualizing a Feature and Its Bounding Box

Let's display one feature and its bounding box on a map to see what it looks like.

In [4]:
from ipyleaflet import Map, GeoJSON

# Pick a longer feature for a more interesting bbox
long_features = sorted(railroads["features"], key=lambda f: len(f["geometry"]["coordinates"]), reverse=True)
f = long_features[2]

bbox = feature_bbox(f)
lon_min, lat_min, lon_max, lat_max = bbox

# Build the bbox as a GeoJSON polygon
bbox_polygon = {
    "type": "Feature",
    "properties": {"name": "bounding box"},
    "geometry": {
        "type": "Polygon",
        "coordinates": [[
            [lon_min, lat_min],
            [lon_max, lat_min],
            [lon_max, lat_max],
            [lon_min, lat_max],
            [lon_min, lat_min],
        ]]
    }
}

center_lat = (lat_min + lat_max) / 2
center_lon = (lon_min + lon_max) / 2

m = Map(center=[center_lat, center_lon], zoom=5)

m.add(GeoJSON(data={"type": "FeatureCollection", "features": [f]},
              style={"color": "#cc3300", "weight": 2}))
m.add(GeoJSON(data={"type": "FeatureCollection", "features": [bbox_polygon]},
              style={"color": "#0066cc", "weight": 1.5, "fillOpacity": 0.05}))
m

Map(center=[63.0397215, 75.576944], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title'…

## Bounding Boxes for the LOD Files

Our LOD output files do not have precomputed `bbox` fields. We will compute them on the fly during culling.

As an optimization preview: we could precompute and store bounding boxes once at pipeline time, then just read the stored values during culling. This is a common real-world pattern.

For now, let's verify the function works on a LOD feature.

In [5]:
lod_path = Path("../../data/lod/railroads_fine.geojson")
with open(lod_path) as f:
    fine = json.load(f)

sample = fine["features"][100]
bbox = feature_bbox(sample)

print("LOD feature bbox:", bbox)
print("Coordinate count:", len(sample["geometry"]["coordinates"]))

LOD feature bbox: [66.311406, 66.714926, 68.935817, 68.190447]
Coordinate count: 26


## Exercise A

Write a function `collection_bbox(features)` that returns the bounding box of an **entire FeatureCollection** — the smallest rectangle that contains all features.

Apply it to each of the four LOD files and compare the results. Do they all cover the same geographic extent?

In [6]:
def collection_bbox(features):
    lon_min = min(feature_bbox(f)[0] for f in features)
    lat_min = min(feature_bbox(f)[1] for f in features)
    lon_max = max(feature_bbox(f)[2] for f in features)
    lat_max = max(feature_bbox(f)[3] for f in features)
    return [lon_min, lat_min, lon_max, lat_max]

lod_files = {
    'coarse': 'railroads_coarse.geojson',
    'medium': 'railroads_medium.geojson',
    'fine': 'railroads_fine.geojson',
    'extra_fine': 'railroads_extra_fine.geojson',
}

bbox_by_level = {}

print(f"{'Level':<12} {'Collection bbox':<42} {'Width':>8} {'Height':>8}")
print('-' * 78)
for name, filename in lod_files.items():
    with open(Path('../../data/lod') / filename) as f:
        fc = json.load(f)
    bbox = collection_bbox(fc['features'])
    bbox_by_level[name] = bbox
    width = bbox[2] - bbox[0]
    height = bbox[3] - bbox[1]
    rounded = [round(v, 2) for v in bbox]
    print(f"{name:<12} {str(rounded):<42} {width:>8.2f} {height:>8.2f}")

all_same = len({tuple(round(v, 6) for v in bbox) for bbox in bbox_by_level.values()}) == 1
print()
print('Same geographic extent across all four files?', all_same)
if not all_same:
    print('The coarse file covers a slightly smaller extent because the scalerank filter removes some low-priority edge features.')


Level        Collection bbox                               Width   Height
------------------------------------------------------------------------------
coarse       [-123.01, -41.48, 150.96, 60.98]             273.98   102.45
medium       [-150.11, -51.89, 179.36, 69.6]              329.47   121.50
fine         [-150.11, -51.89, 179.36, 69.6]              329.47   121.50
extra_fine   [-150.11, -51.9, 179.36, 69.6]               329.47   121.50

Same geographic extent across all four files? False
The coarse file covers a slightly smaller extent because the scalerank filter removes some low-priority edge features.


## Exercise B

Find the **5 features with the largest bounding box area** in the fine LOD file.

Bounding box area = `(lon_max - lon_min) * (lat_max - lat_min)`.

Print each one's bbox area and its `category` property. Do the results make geographic sense?

In [7]:
def bbox_area(bbox):
    lon_min, lat_min, lon_max, lat_max = bbox
    return max(0, lon_max - lon_min) * max(0, lat_max - lat_min)

ranked = []
for idx, feature in enumerate(fine['features']):
    bbox = feature_bbox(feature)
    props = feature['properties']
    ranked.append((bbox_area(bbox), idx, props.get('category'), props.get('rwdb_rr_id'), bbox))

top_five = sorted(ranked, reverse=True)[:5]

print(f"{'Rank':<6} {'Area':>10} {'Feature #':>10} {'Category':>10} {'rwdb_rr_id':>12} {'bbox':>30}")
print('-' * 100)
for rank, (area, idx, category, rrid, bbox) in enumerate(top_five, start=1):
    print(f"{rank:<6} {area:>10.3f} {idx:>10} {str(category):>10} {str(rrid):>12} {str([round(v, 3) for v in bbox]):>30}")

print()
print('Yes, the biggest bounding boxes belong to very long railroad features, so their large geographic footprints make sense.')


Rank         Area  Feature #   Category   rwdb_rr_id                           bbox
----------------------------------------------------------------------------------------------------
1          29.312      25411          0            0 [90.609, 29.645, 94.943, 36.409]
2          18.649      25409          0            0 [132.256, -23.55, 134.323, -14.531]
3          17.704      24238          3        24239 [-64.094, -26.192, -58.157, -23.211]
4          17.387       4803          2         4804 [14.022, 54.803, 20.0, 57.711]
5          12.355      23589          2        23590 [-49.138, -5.492, -44.372, -2.899]

Yes, the biggest bounding boxes belong to very long railroad features, so their large geographic footprints make sense.


## Check Your Understanding

Two different railroad features can have identical bounding boxes even though they follow completely different paths.

Describe a scenario where this happens — what would the two features look like? And does this cause any problem for our culling system?

Two rail lines could share the same westernmost, easternmost, northernmost, and southernmost coordinates while taking different shapes inside that box, such as one nearly straight diagonal and one S-shaped route between the same extremes. That is acceptable for our culling system because the bounding box is only a fast first-pass reject test; the consequence is a few extra candidates survive the cull, not that visible features are missed.


## Next

In [01 — Intersection Test](./01-Intersection_Test.ipynb), we write the function that checks whether a feature's bounding box overlaps the current viewport.